In [18]:
from dotenv import load_dotenv
load_dotenv()
import os
groq_api_key = os.getenv("GROQ_API_KEY")
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")
pinecone_api_key = os.getenv("PINECONE_API_KEY")

In [19]:
from langchain_groq import ChatGroq
model = ChatGroq(model = "openai/gpt-oss-120b",groq_api_key = groq_api_key)
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001D60B20E9E0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001D60CBE4A30>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [20]:
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader("Attention.pdf")
docs = loader.load()
docs 

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'Attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗ †\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser ∗\nGoogle Brain\nlukasz

In [21]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8050.64it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [22]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone

pc = Pinecone(api_key=pinecone_api_key)

index = pc.Index("researchchatbot")

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
splits = splitter.split_documents(docs)

vectorstoredb = PineconeVectorStore(
    index=index, 
    embedding=embeddings
)

vectorstoredb.add_documents(documents=splits)


['7277168b-5454-4f14-86bc-74b0b5b334c2',
 '7d9e536f-efa3-4bcc-8026-4d6da945bf15',
 'a94c7ee8-58e4-433a-b16a-22d15b30032c',
 'bd112348-f9f8-43a9-9e5a-5399de56f92e',
 'bd3168f7-3a79-49dc-875b-e800f36f39e2',
 '7308bcdb-0101-413f-8fb8-a41b2e1cb950',
 '4d5a80a3-9c15-40fa-8bb3-34c507fdb22a',
 'e1bc5e90-a763-4f47-b9a2-069f03bfb6cb',
 'a302dd61-791f-4b75-942d-0c7f762e41af',
 'ff00fdd2-af6e-4ace-872b-f87b5b563127',
 '88f670ff-7e88-49ef-88b5-01580b89116e',
 '5ac3e1d3-e907-4658-9ddd-e3bb87c34d9f',
 'ce93c0f6-8f2d-4ed9-8d33-625c591e8b0b',
 '88a428a8-05b3-4e5a-9c80-7516f53586d1',
 '274395b6-ff2f-41fb-a504-7b6e485266ee',
 '03d93c4d-81e4-494e-8620-93c899de724e',
 'a8ae91bd-ee88-4e84-af7b-a3fb962d9c98',
 '219a2cf5-5b1a-41d8-b33f-2264aed46f44',
 'cfe5c2a7-a38d-482d-8ec9-0e847d0df3f5',
 'c2505c31-2454-45aa-a943-b59feb6df3be',
 '3aac0ef2-24ad-4df8-a8ee-37e9c717946f',
 '4089e5e0-9f33-4514-8c50-3061f1965446',
 '8c86b041-de33-4a61-bc39-0f211571f894',
 '77bf3e23-5d8c-4e73-babb-063635b55929',
 'f616ba47-bfb8-

In [23]:
retriever = vectorstoredb.as_retriever()
retriever

VectorStoreRetriever(tags=['PineconeVectorStore', 'HuggingFaceEmbeddings'], vectorstore=<langchain_pinecone.vectorstores.PineconeVectorStore object at 0x000001D601F43940>, search_kwargs={})

In [24]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are "ResearchAI", an expert research assistant. Your goal is to analyze the provided context from research papers/documents and answer the user's question with 100% accuracy.

---
### 📜 CONTEXT FROM DOCUMENTS:
{context}

---
### 👤 USER QUESTION:
{input}

---
### 🛠 INSTRUCTIONS:
1. **Strict Grounding:** Answer ONLY based on the provided context. Do not use outside knowledge.
2. **Handle Uncertainty:** If the answer is not present in the context, say: "I'm sorry, but the provided documents do not contain information to answer this specific question." Do not hallucinate.
3. **Citations:** When you quote or reference a point, mention the source/page number if available in the metadata.
4. **Structure:** Use bullet points and bold text for key terms to make the research findings easy to read.
5. **Tone:** Maintain a professional, academic, and objective tone.

### 📝 YOUR RESPONSE:
""")

In [25]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

document_chain = create_stuff_documents_chain(model,prompt)
retrieval_chain = create_retrieval_chain(retriever,document_chain)


In [26]:
query = "What is the Transformer architecture?"
response = retrieval_chain.invoke({"input": query})
print(response["answer"])

**Transformer Architecture (as described in the provided documents)**  

- **Overall Structure**
  - The model consists of two symmetric halves: an **encoder** and a **decoder** (Figure 1) 【Context】.  
  - Both sides are built from **stacked self‑attention layers** combined with **point‑wise, fully‑connected feed‑forward layers** 【Context】.  

- **Encoder**
  - Composed of **N = 6 identical layers** 【Context】.  
  - Each encoder layer contains **two sub‑layers**:
    1. **Multi‑head self‑attention mechanism**.  
    2. **Position‑wise fully connected feed‑forward network**.  
  - A **residual connection** surrounds each sub‑layer, followed by **layer normalization**:  
    `LayerNorm(x + Sublayer(x))` 【Context】.  
  - All sub‑layers and the embedding layers produce outputs of dimension **d_model = 512** to enable the residual connections 【Context】.  

- **Decoder**
  - Mirrors the encoder’s overall design (stacked layers with self‑attention and feed‑forward components) as shown in the 

In [1]:
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent

c:\Users\vishankh\Desktop\RAG PROJECT\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
help(AgentExecutor)

Help on class AgentExecutor in module langchain_classic.agents.agent:

class AgentExecutor(langchain_classic.chains.base.Chain)
 |  AgentExecutor(*args: Any, name: str | None = None, memory: langchain_classic.base_memory.BaseMemory | None = None, callbacks: list[langchain_core.callbacks.base.BaseCallbackHandler] | langchain_core.callbacks.base.BaseCallbackManager | None = None, verbose: bool = <factory>, tags: list[str] | None = None, metadata: dict[str, typing.Any] | None = None, callback_manager: langchain_core.callbacks.base.BaseCallbackManager | None = None, agent: langchain_classic.agents.agent.BaseSingleActionAgent | langchain_classic.agents.agent.BaseMultiActionAgent | langchain_core.runnables.base.Runnable, tools: collections.abc.Sequence[langchain_core.tools.base.BaseTool], return_intermediate_steps: bool = False, max_iterations: int | None = 15, max_execution_time: float | None = None, early_stopping_method: str = 'force', handle_parsing_errors: bool | str | collections.abc.C

In [3]:
help(create_tool_calling_agent)

Help on function create_tool_calling_agent in module langchain_classic.agents.tool_calling_agent.base:

create_tool_calling_agent(llm: langchain_core.language_models.base.BaseLanguageModel, tools: collections.abc.Sequence[langchain_core.tools.base.BaseTool], prompt: langchain_core.prompts.chat.ChatPromptTemplate, *, message_formatter: collections.abc.Callable[[collections.abc.Sequence[tuple[langchain_core.agents.AgentAction, str]]], list[langchain_core.messages.base.BaseMessage]] = <function format_to_tool_messages at 0x000001E1E9A63B50>) -> langchain_core.runnables.base.Runnable
    Create an agent that uses tools.
    
    Args:
        llm: LLM to use as the agent.
        tools: Tools this agent has access to.
        prompt: The prompt to use. See Prompt section below for more on the expected
            input variables.
        message_formatter: Formatter function to convert (AgentAction, tool output)
            tuples into FunctionMessages.
    
    Returns:
        A Runnable